# sld-poc — Stage 2: latent diffusion (JAX / TPU)

Trains a self-conditioned Gaussian-diffusion denoiser over the codec's latent grids, then **generates**, **infills**, and **self-corrects**.

**Requires Stage A finished** (`codec.pkl` + `latent_stats.npz` in your Drive workdir).

**Before running:** Runtime → **TPU**. Code runs *in-kernel* (the TPU is single-process).

## 1. Confirm the TPU

In [ ]:
import jax, jax.numpy as jnp
print(jax.devices()); print(jax.default_backend())

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo + install deps
Make the repo **public**, or add a GitHub token to clone.

In [ ]:
import os
REPO_URL = 'https://github.com/mccooking/sld-poc.git'
REPO_DIR = '/content/sld-poc'
DATA_DIR = '/content/drive/MyDrive/sld-poc-data'      # same workdir as Stage A
if os.path.exists(REPO_DIR):
    !cd $REPO_DIR && git pull
else:
    !git clone $REPO_URL $REPO_DIR
!pip -q install -r $REPO_DIR/requirements.txt
assert os.path.exists(f'{DATA_DIR}/codec.pkl'), 'Run Stage A (01_codec) to completion first'
print('ready:', DATA_DIR)

## 4. Build latent grids (once)
Encodes TinyStories chunks into standardized latent grids using the frozen Stage A codec. Stored on local disk (rebuilds automatically after a disconnect).

In [ ]:
import sys, importlib
SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)
import diffusion; importlib.reload(diffusion)
diffusion.build_latents(diffusion.paths(DATA_DIR))

## 5. Train the denoiser  ⟵  re-run after any disconnect (auto-resumes)
Watch `mse` trend down (it's noisy — that's normal). Continuous text diffusion is finicky; if generation looks rough, re-run this to train longer.

In [ ]:
import sys, importlib
SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)
import diffusion; importlib.reload(diffusion)
diffusion.train_diff(diffusion.paths(DATA_DIR))

## 6a. Generation + speed

In [ ]:
import sys, importlib
SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)
import diffusion; importlib.reload(diffusion)
diffusion.gen(diffusion.paths(DATA_DIR))

## 6b. Infilling — fix the ends, generate the middle *(AR can't do this)*

In [ ]:
import sys, importlib
SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)
import diffusion; importlib.reload(diffusion)
diffusion.infill(diffusion.paths(DATA_DIR))

## 6c. Self-correction — corrupt one latent, repair from context *(AR can't do this)*

In [ ]:
import sys, importlib
SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)
import diffusion; importlib.reload(diffusion)
diffusion.selfcorrect(diffusion.paths(DATA_DIR))